Step 1: Import all dependencies to be used. We will be using MediaPipe's hand tracker to draw and track our hand positions, then use dynamic time warping to calculate the distance between the tracked hand and all the entries in our dataset.

This portfolio version loads pre-extracted hand landmarks from pickle files in `data/dataset` instead of processing reference videos.

In [4]:
import pandas as pd
import cv2
import mediapipe as mp
import numpy as np
import os
import pickle as pkl
from fastdtw import fastdtw
from matplotlib import pyplot as plt
mp_drawing = mp.solutions.drawing_utils
mp_hands = mp.solutions.hands

Defining our functions:
1. load_array: used to load an array of extracted landmarks from our reference dataset so that we can add it to our dataframe
2. getLabel: extracts hand information such as left/right, and coordinates of every joint. Used to draw the landmark and extract landmarks from each frame
3. load_dataset_from_pickles: discovers all reference signs from pre-saved pickle files in `data/dataset`
4. load_reference_signs: create our dataframe containing name, left_hand_sign, right_hand_sign, and distance

In [5]:
def load_array(path):
    file = open(path, "rb")
    arr = pkl.load(file)
    file.close()
    return np.array(arr)

def getLabel(index, hand, results, resolution):
    output = None
    if index == 0:
        label = results.multi_handedness[0].classification[0].label
        coords_comp = np.array([(hand.landmark[mp_hands.HandLandmark.WRIST].x, hand.landmark[mp_hands.HandLandmark.WRIST].y),
                              (hand.landmark[mp_hands.HandLandmark.THUMB_CMC].x, hand.landmark[mp_hands.HandLandmark.THUMB_CMC].y),
                              (hand.landmark[mp_hands.HandLandmark.THUMB_MCP].x, hand.landmark[mp_hands.HandLandmark.THUMB_MCP].y),
                              (hand.landmark[mp_hands.HandLandmark.THUMB_IP].x, hand.landmark[mp_hands.HandLandmark.THUMB_IP].y),
                              (hand.landmark[mp_hands.HandLandmark.THUMB_TIP].x, hand.landmark[mp_hands.HandLandmark.THUMB_TIP].y),
                              (hand.landmark[mp_hands.HandLandmark.INDEX_FINGER_MCP].x, hand.landmark[mp_hands.HandLandmark.INDEX_FINGER_MCP].y),
                              (hand.landmark[mp_hands.HandLandmark.INDEX_FINGER_PIP].x, hand.landmark[mp_hands.HandLandmark.INDEX_FINGER_PIP].y),
                              (hand.landmark[mp_hands.HandLandmark.INDEX_FINGER_DIP].x, hand.landmark[mp_hands.HandLandmark.INDEX_FINGER_DIP].y),
                              (hand.landmark[mp_hands.HandLandmark.INDEX_FINGER_TIP].x, hand.landmark[mp_hands.HandLandmark.INDEX_FINGER_TIP].y),
                              (hand.landmark[mp_hands.HandLandmark.MIDDLE_FINGER_MCP].x, hand.landmark[mp_hands.HandLandmark.MIDDLE_FINGER_MCP].y),
                              (hand.landmark[mp_hands.HandLandmark.MIDDLE_FINGER_PIP].x, hand.landmark[mp_hands.HandLandmark.MIDDLE_FINGER_PIP].y),
                              (hand.landmark[mp_hands.HandLandmark.MIDDLE_FINGER_DIP].x, hand.landmark[mp_hands.HandLandmark.MIDDLE_FINGER_DIP].y),
                              (hand.landmark[mp_hands.HandLandmark.MIDDLE_FINGER_TIP].x, hand.landmark[mp_hands.HandLandmark.MIDDLE_FINGER_TIP].y),
                               (hand.landmark[mp_hands.HandLandmark.RING_FINGER_MCP].x, hand.landmark[mp_hands.HandLandmark.RING_FINGER_MCP].y),
                               (hand.landmark[mp_hands.HandLandmark.RING_FINGER_PIP].x, hand.landmark[mp_hands.HandLandmark.RING_FINGER_PIP].y),
                               (hand.landmark[mp_hands.HandLandmark.RING_FINGER_DIP].x, hand.landmark[mp_hands.HandLandmark.RING_FINGER_DIP].y),
                               (hand.landmark[mp_hands.HandLandmark.RING_FINGER_TIP].x, hand.landmark[mp_hands.HandLandmark.RING_FINGER_TIP].y),
                               (hand.landmark[mp_hands.HandLandmark.PINKY_MCP].x, hand.landmark[mp_hands.HandLandmark.PINKY_MCP].y),
                               (hand.landmark[mp_hands.HandLandmark.PINKY_PIP].x, hand.landmark[mp_hands.HandLandmark.PINKY_PIP].y),
                               (hand.landmark[mp_hands.HandLandmark.PINKY_DIP].x, hand.landmark[mp_hands.HandLandmark.PINKY_DIP].y),
                               (hand.landmark[mp_hands.HandLandmark.PINKY_TIP].x, hand.landmark[mp_hands.HandLandmark.PINKY_TIP].y)])
        coords = tuple(np.multiply(
                        np.array((hand.landmark[mp_hands.HandLandmark.WRIST].x, hand.landmark[mp_hands.HandLandmark.WRIST].y)),
                        resolution).astype(int))

        output = label, coords, coords_comp
        return output
    
    if index == 1:
        label = results.multi_handedness[1].classification[0].label
        coords_comp = np.array([(hand.landmark[mp_hands.HandLandmark.WRIST].x, hand.landmark[mp_hands.HandLandmark.WRIST].y),
                              (hand.landmark[mp_hands.HandLandmark.THUMB_CMC].x, hand.landmark[mp_hands.HandLandmark.THUMB_CMC].y),
                              (hand.landmark[mp_hands.HandLandmark.THUMB_MCP].x, hand.landmark[mp_hands.HandLandmark.THUMB_MCP].y),
                              (hand.landmark[mp_hands.HandLandmark.THUMB_IP].x, hand.landmark[mp_hands.HandLandmark.THUMB_IP].y),
                              (hand.landmark[mp_hands.HandLandmark.THUMB_TIP].x, hand.landmark[mp_hands.HandLandmark.THUMB_TIP].y),
                              (hand.landmark[mp_hands.HandLandmark.INDEX_FINGER_MCP].x, hand.landmark[mp_hands.HandLandmark.INDEX_FINGER_MCP].y),
                              (hand.landmark[mp_hands.HandLandmark.INDEX_FINGER_PIP].x, hand.landmark[mp_hands.HandLandmark.INDEX_FINGER_PIP].y),
                              (hand.landmark[mp_hands.HandLandmark.INDEX_FINGER_DIP].x, hand.landmark[mp_hands.HandLandmark.INDEX_FINGER_DIP].y),
                              (hand.landmark[mp_hands.HandLandmark.INDEX_FINGER_TIP].x, hand.landmark[mp_hands.HandLandmark.INDEX_FINGER_TIP].y),
                              (hand.landmark[mp_hands.HandLandmark.MIDDLE_FINGER_MCP].x, hand.landmark[mp_hands.HandLandmark.MIDDLE_FINGER_MCP].y),
                              (hand.landmark[mp_hands.HandLandmark.MIDDLE_FINGER_PIP].x, hand.landmark[mp_hands.HandLandmark.MIDDLE_FINGER_PIP].y),
                              (hand.landmark[mp_hands.HandLandmark.MIDDLE_FINGER_DIP].x, hand.landmark[mp_hands.HandLandmark.MIDDLE_FINGER_DIP].y),
                              (hand.landmark[mp_hands.HandLandmark.MIDDLE_FINGER_TIP].x, hand.landmark[mp_hands.HandLandmark.MIDDLE_FINGER_TIP].y),
                               (hand.landmark[mp_hands.HandLandmark.RING_FINGER_MCP].x, hand.landmark[mp_hands.HandLandmark.RING_FINGER_MCP].y),
                               (hand.landmark[mp_hands.HandLandmark.RING_FINGER_PIP].x, hand.landmark[mp_hands.HandLandmark.RING_FINGER_PIP].y),
                               (hand.landmark[mp_hands.HandLandmark.RING_FINGER_DIP].x, hand.landmark[mp_hands.HandLandmark.RING_FINGER_DIP].y),
                               (hand.landmark[mp_hands.HandLandmark.RING_FINGER_TIP].x, hand.landmark[mp_hands.HandLandmark.RING_FINGER_TIP].y),
                               (hand.landmark[mp_hands.HandLandmark.PINKY_MCP].x, hand.landmark[mp_hands.HandLandmark.PINKY_MCP].y),
                               (hand.landmark[mp_hands.HandLandmark.PINKY_PIP].x, hand.landmark[mp_hands.HandLandmark.PINKY_PIP].y),
                               (hand.landmark[mp_hands.HandLandmark.PINKY_DIP].x, hand.landmark[mp_hands.HandLandmark.PINKY_DIP].y),
                               (hand.landmark[mp_hands.HandLandmark.PINKY_TIP].x, hand.landmark[mp_hands.HandLandmark.PINKY_TIP].y)])
        coords = tuple(np.multiply(
                        np.array((hand.landmark[mp_hands.HandLandmark.WRIST].x, hand.landmark[mp_hands.HandLandmark.WRIST].y)),
                        resolution).astype(int))

        output = label, coords, coords_comp
        return output

def load_dataset_from_pickles():
    """Load reference sign names from pre-extracted pickle files in data/dataset."""
    dataset = [
        file_name.replace(".pickle", "").replace("lh_", "")
        for root, dirs, files in os.walk(os.path.join("data", "dataset"))
        for file_name in files
        if file_name.endswith(".pickle") and file_name.startswith("lh_")
    ]
    print(f"Loaded {len(dataset)} reference signs from pickle files")
    return dataset

def load_reference_signs(videos):
    reference_signs = {"name": [], "left_sign_model": [], "right_sign_model": [], "distance": []}
    for video_name in videos:
        sign_name = video_name.split(" ")[0]
        path = os.path.join("data", "dataset", sign_name, video_name)

        left_hand_list = load_array(os.path.join(path, f"lh_{video_name}.pickle"))
        right_hand_list = load_array(os.path.join(path, f"rh_{video_name}.pickle"))

        reference_signs["name"].append(sign_name)
        reference_signs["left_sign_model"].append(left_hand_list)
        reference_signs["right_sign_model"].append(right_hand_list)
        reference_signs["distance"].append(0)
    
    reference_signs = pd.DataFrame(reference_signs, dtype=object)
    print(
        f'Dictionary count: {reference_signs[["name", "left_sign_model", "right_sign_model"]].groupby(["name"]).count()}'
    )
    return reference_signs

In [6]:
videos = load_dataset_from_pickles()

Loaded 578 reference signs from pickle files


In [7]:
reference_signs = load_reference_signs(videos)

Dictionary count:            left_sign_model  right_sign_model
name                                        
hundred                153               153
mother                 157               157
ninetytwo               42                42
relax                  166               166
salute                  60                60


In [ ]:
reference_signs.to_csv('df.csv')

# The cell below records your sign language. Press "r" to start recording your sign, and press "q" once you are done to quit the window

In [27]:
cap = cv2.VideoCapture(0) #record from webcam
landmarks_current = {'right':[], 'left':[]} #extracted landmarks to be saved here
resolution = [640,480] #resolution of the webcam
cap_len = 50 #max length of frames to record
is_recording = False
with mp_hands.Hands(min_detection_confidence=0.8, min_tracking_confidence=0.5) as hands: 
    while cap.isOpened():
        ret, frame = cap.read()
        if ret == True:
            image = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            image = cv2.flip(image, 1)
            image.flags.writeable = False
            results = hands.process(image)
            image.flags.writeable = True
            image = cv2.cvtColor(image, cv2.COLOR_RGB2BGR)
            if results.multi_hand_landmarks: #if hand has been recognized
                for num, hand in enumerate(results.multi_hand_landmarks):
                    mp_drawing.draw_landmarks(image, hand, mp_hands.HAND_CONNECTIONS, 
                                        mp_drawing.DrawingSpec(color=(121, 22, 76), thickness=2, circle_radius=4),
                                        mp_drawing.DrawingSpec(color=(250, 44, 250), thickness=2, circle_radius=2)
                                         )
                    if getLabel(num, hand, results, resolution): #if hand is present
                        text, coord, coords_comp = getLabel(num, hand, results, resolution) #extract hand, coordinates of wrist, coordinates of joints
                        if (is_recording == True) and ((len(landmarks_current['right']) < cap_len) or len(landmarks_current['left']) < cap_len): #if recording
                            if text == 'Right':
                                landmarks_current['right'].append(coords_comp) #add right hand joint coordinates to dict
                            if text == 'Left':
                                landmarks_current['left'].append(coords_comp) #add left hand joint coordinates to dict
                        cv2.putText(image, text, coord, cv2.FONT_HERSHEY_SIMPLEX, 1, (255,255,255), 2, cv2.LINE_AA) # write name of hand
            cv2.imshow('Hand Tracking', image)
            if cv2.waitKey(10) & 0xFF == ord('q'): #quit
                break
            elif cv2.waitKey(10) & 0xFF == ord('r'): #press r to record
                is_recording = True
            if (len(landmarks_current['right']) == cap_len) or len(landmarks_current['left']) == cap_len:
                is_recording = False
        else:
            break
cap.release()
cv2.destroyAllWindows()

In [28]:
print(len(landmarks_current['right'])) #number of frames recorded per hand
print(len(landmarks_current['left']))
landmarks_current

0
16


{'right': [],
 'left': [array([[0.33139244, 0.86272496],
         [0.38875934, 0.7986052 ],
         [0.41075331, 0.72409153],
         [0.42584935, 0.66637868],
         [0.42997849, 0.61546332],
         [0.32370833, 0.61718583],
         [0.32821858, 0.5304755 ],
         [0.33275485, 0.4623878 ],
         [0.33700904, 0.40758914],
         [0.29595715, 0.63173777],
         [0.39550605, 0.61277354],
         [0.43184817, 0.68345332],
         [0.43940529, 0.73437667],
         [0.28743863, 0.66798156],
         [0.39058635, 0.6686421 ],
         [0.40975136, 0.73122329],
         [0.40634596, 0.77318686],
         [0.2917116 , 0.71371043],
         [0.3730588 , 0.70762849],
         [0.3946532 , 0.749111  ],
         [0.39624283, 0.78044063]]),
  array([[0.32637307, 0.85589373],
         [0.39297584, 0.77850544],
         [0.41631347, 0.70306504],
         [0.43110189, 0.64493018],
         [0.43820241, 0.59059352],
         [0.33190712, 0.60714775],
         [0.33790025, 0.5156335

In [29]:
reference_signs #our dataframe

,name,left_sign_model,right_sign_model,distance
0,hundred,"[[[0.44234949350357056, 0.9823299646377563], [...",[],0
1,hundred,"[[[0.43554818630218506, 0.9323959350585938], [...",[],0
2,hundred,"[[[0.3997741937637329, 0.9753984808921814], [0...",[],0
3,hundred,"[[[0.4029337465763092, 1.0132941007614136], [0...",[],0
4,hundred,"[[[0.39897865056991577, 1.019285798072815], [0...",[],0
...,...,...,...,...
573,salute,"[[[0.13372300565242767, 0.7204000949859619], [...",[],0
574,salute,"[[[0.23447486758232117, 0.8026021718978882], [...",[],0
575,salute,"[[[0.17358015477657318, 0.9119427800178528], [...",[],0
576,salute,"[[[0.16415748000144958, 0.9310551881790161], [...",[],0


The cell below is added because sometimes a hand is incorrectly detected. It will remove the extracted landmarks if there are less than 9 frames for each hand. Number of frames can be tuned for better performance.

In [30]:
if len(landmarks_current['right']) < 9:
    landmarks_current['right'] = []
if len(reference_signs['left_sign_model']) < 9:
    reference_signs['left_sign_model'] = []
if len(reference_signs['right_sign_model']) < 9:
    reference_signs['right_sign_model'] = []
if len(landmarks_current['left']) < 9:
    landmarks_current['left'] = []

This is where Dynamic Time Warping is used to calculate the distance between the recorded sign and all the signs in the dataframe

In [31]:
has_left_hand_landmarks_current = np.sum(landmarks_current['left']) != 0 #True if left hand has been detected, false otherwise
has_right_hand_landmarks_current = np.sum(landmarks_current['right']) != 0
for index in range(len(reference_signs)):
    has_left_hand_ref = np.sum(reference_signs['left_sign_model'][index]) != 0
    has_right_hand_ref = np.sum(reference_signs['right_sign_model'][index]) != 0
    if has_left_hand_landmarks_current == has_left_hand_ref and has_right_hand_landmarks_current == has_right_hand_ref:
        reference_signs['distance'][index] += fastdtw(landmarks_current["left"], reference_signs["left_sign_model"][index])[0]
        reference_signs['distance'][index] += fastdtw(landmarks_current["right"], reference_signs["right_sign_model"][index])[0]
    else:
        reference_signs['distance'][index] = np.inf
        
#If there are the same number of hands in our recorded sign and the reference sign, then we will compute DTW. Otherwise, we
#will set the distance to infinity so that it goes to the bottom of the list

C:\Users\Mohannad\AppData\Local\Temp\ipykernel_14788\2494137548.py:7: FutureWarning: ChainedAssignmentError: behaviour will change in pandas 3.0!
You are setting values through chained assignment. Currently this works in certain cases, but when using Copy-on-Write (which will become the default behaviour in pandas 3.0) this will never work to update the original DataFrame or Series, because the intermediate object on which we are setting values will behave as a copy.
A typical example is when you are setting values in a column of a DataFrame, like:

df["col"][row_indexer] = value

Use `df.loc[row_indexer, "col"] = values` instead, to perform the assignment in a single step and ensure this keeps updating the original `df`.

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

  reference_signs['distance'][index] += fastdtw(landmarks_current["left"], reference_signs["left_sign_model"][index])[0]
C:\Use

In [32]:
pred_df = reference_signs[reference_signs.distance==reference_signs.distance.min()]
#find the predicted sign. It will be the row where the distance is minimum

In [33]:
if pred_df['distance'].values[0] > 150: #distance threshold. Can be tuned for better performance
    print('العلامة المتوقعة: مجهول')
else:
    print(f"العلامة المتوقعة: {pred_df['name'].values[0]}" )

العلامة المتوقعة: mother


In [34]:
reference_signs.sort_values('distance').head(15) #our dataframe, sorted by lowest distance with recorded sign

,name,left_sign_model,right_sign_model,distance
156,mother,"[[[0.3839678466320038, 0.8771167993545532], [0...",[],23.573205
163,mother,"[[[0.3885917365550995, 0.9004017114639282], [0...",[],24.498931
158,mother,"[[[0.38037586212158203, 0.9010438323020935], [...",[],25.111500
193,mother,"[[[0.28673961758613586, 0.978580117225647], [0...",[],25.478135
189,mother,"[[[0.2936985194683075, 0.9754341840744019], [0...",[],26.870752
160,mother,"[[[0.3831060528755188, 0.8759055137634277], [0...",[],27.108327
171,mother,"[[[0.3551342487335205, 0.90412837266922], [0.3...",[],27.257944
161,mother,"[[[0.37486693263053894, 0.8706103563308716], [...",[],28.718660
164,mother,"[[[0.3784366548061371, 0.8803051114082336], [0...",[],29.193280
178,mother,"[[[0.3647693395614624, 0.8840281367301941], [0...",[],29.416806


In [25]:
reference_signs['distance'] = 0 #reset distance column before recording next sign